# Worthly — Laptop Price Prediction

Final machine-learning notebook for the **Worthly** Django project.

This notebook covers dataset inspection, preprocessing, feature engineering, Random Forest training, evaluation, cross-validation, visual checks, and model export.

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)

## 2. Load the Dataset

Place `laptop_data.csv` in the same folder as this notebook before running all cells.

In [ ]:
df = pd.read_csv("laptop_data.csv")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("Dataset shape:", df.shape)
display(df.head())

## 3. Dataset Check

In [ ]:
print("Columns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

## 4. Feature Engineering

In [ ]:
df["Ram"] = df["Ram"].astype(str).str.replace("GB", "", regex=False).astype(int)
df["Weight"] = df["Weight"].astype(str).str.replace("kg", "", regex=False).astype(float)

df["Touchscreen"] = df["ScreenResolution"].astype(str).str.contains(
    "Touchscreen", case=False, na=False
).astype(int)

df["Ips"] = df["ScreenResolution"].astype(str).str.contains(
    "IPS", case=False, na=False
).astype(int)

resolution = df["ScreenResolution"].astype(str).str.extract(r"(\d{3,4})x(\d{3,4})")
df["X_res"] = pd.to_numeric(resolution[0], errors="coerce")
df["Y_res"] = pd.to_numeric(resolution[1], errors="coerce")
df["ppi"] = np.sqrt(df["X_res"]**2 + df["Y_res"]**2) / df["Inches"]

def cpu_brand(cpu):
    cpu = str(cpu)
    if "Intel Core i7" in cpu:
        return "Intel Core i7"
    if "Intel Core i5" in cpu:
        return "Intel Core i5"
    if "Intel Core i3" in cpu:
        return "Intel Core i3"
    if "Intel" in cpu:
        return "Other Intel Processor"
    return "AMD Processor"

df["Cpu brand"] = df["Cpu"].apply(cpu_brand)
df["Gpu brand"] = df["Gpu"].astype(str).str.split().str[0]
df = df[df["Gpu brand"].isin(["AMD", "Intel", "Nvidia"])].copy()

def os_group(value):
    value = str(value)
    if value in ["Windows 10", "Windows 7", "Windows 10 S"]:
        return "Windows"
    if value in ["macOS", "Mac OS X"]:
        return "Mac"
    return "Others/No OS/Linux"

df["os"] = df["OpSys"].apply(os_group)

print("Rows after initial feature engineering:", len(df))

In [ ]:
def storage_to_gb(value):
    value = str(value).strip()
    if "TB" in value:
        return int(float(value.replace("TB", "").strip()) * 1000)
    if "GB" in value:
        return int(float(value.replace("GB", "").strip()))
    return 0

def extract_storage(memory):
    hdd, ssd = 0, 0
    for part in str(memory).split("+"):
        part = part.strip()
        if "HDD" in part:
            hdd += storage_to_gb(part.replace("HDD", ""))
        elif "SSD" in part:
            ssd += storage_to_gb(part.replace("SSD", ""))
    return pd.Series([hdd, ssd])

df[["HDD", "SSD"]] = df["Memory"].apply(extract_storage)

print("Final processed rows:", len(df))
display(df.head())

## 5. Features and Target

The model uses the same 12 inputs expected by the Worthly Django prediction service. Price is log-transformed during training and converted back with `np.exp()` when displaying a prediction.

In [ ]:
FEATURES = [
    "Company", "TypeName", "Ram", "Weight", "Touchscreen", "Ips",
    "ppi", "HDD", "SSD", "Cpu brand", "Gpu brand", "os"
]

X = df[FEATURES].copy()
y = np.log(df["Price"].astype(float))

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())

## 6. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=2
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 7. Preprocessing and Random Forest

In [ ]:
categorical_features = [
    "Company", "TypeName", "Cpu brand", "Gpu brand", "os"
]

numerical_features = [
    "Ram", "Weight", "Touchscreen", "Ips", "ppi", "HDD", "SSD"
]

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("numerical", "passthrough", numerical_features),
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=500,
        random_state=2,
        n_jobs=-1
    )),
])

pipeline.fit(X_train, y_train)
print("Training complete.")

## 8. Holdout Evaluation

In [ ]:
log_predictions = pipeline.predict(X_test)

actual_prices = np.exp(y_test.to_numpy())
predicted_prices = np.exp(log_predictions)

r2 = r2_score(y_test, log_predictions)
log_mae = mean_absolute_error(y_test, log_predictions)
price_mae = mean_absolute_error(actual_prices, predicted_prices)
price_rmse = np.sqrt(mean_squared_error(actual_prices, predicted_prices))

print(f"R²:        {r2:.4f}")
print(f"Log MAE:   {log_mae:.4f}")
print(f"Price MAE: Rs. {price_mae:,.2f}")
print(f"RMSE:      Rs. {price_rmse:,.2f}")

**Note:** R² is not percentage accuracy. It measures how much variation in the transformed target is explained by the model.

## 9. 5-Fold Cross-Validation

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=2)

cv_r2 = cross_val_score(
    pipeline, X, y, cv=cv, scoring="r2", n_jobs=-1
)

print("Fold R² scores:", np.round(cv_r2, 4))
print(f"Mean CV R²: {cv_r2.mean():.4f}")
print(f"Standard deviation: {cv_r2.std():.4f}")

## 10. Actual vs Predicted Prices

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(actual_prices, predicted_prices, alpha=0.65)

low = min(actual_prices.min(), predicted_prices.min())
high = max(actual_prices.max(), predicted_prices.max())
plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Laptop Prices")
plt.tight_layout()
plt.show()

## 11. Residual Check

In [ ]:
residuals = actual_prices - predicted_prices

plt.figure(figsize=(7, 5))
plt.scatter(predicted_prices, residuals, alpha=0.65)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Prediction Residuals")
plt.tight_layout()
plt.show()

print(f"Median absolute error: Rs. {np.median(np.abs(residuals)):,.2f}")

## 12. Example Test Predictions

In [ ]:
results = X_test.copy()
results["Actual Price"] = actual_prices
results["Predicted Price"] = predicted_prices
results["Absolute Error"] = np.abs(actual_prices - predicted_prices)

display(
    results[[
        "Company", "TypeName", "Ram", "SSD", "Cpu brand",
        "Actual Price", "Predicted Price", "Absolute Error"
    ]].head(10).round(2)
)

## 13. Train the Final Deployment Model

After evaluation, the deployment model is trained using all processed records.

In [ ]:
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=500,
        random_state=2,
        n_jobs=-1
    )),
])

final_model.fit(X, y)
print("Final model trained on", len(X), "records.")

## 14. Export the Pickle Model

In [ ]:
with open("laptop_price_model.pkl", "wb") as file:
    pickle.dump(final_model, file)

print("Saved laptop_price_model.pkl")

## 15. Verify the Saved Model

In [ ]:
with open("laptop_price_model.pkl", "rb") as file:
    loaded_model = pickle.load(file)

sample = X.iloc[[0]]
sample_prediction = np.exp(loaded_model.predict(sample)[0])

display(sample)
print(f"Predicted price: Rs. {sample_prediction:,.2f}")
print(f"Actual price:    Rs. {df.iloc[0]['Price']:,.2f}")

## Conclusion

The final Worthly model uses a Random Forest regression pipeline with one-hot encoding for categorical inputs. It is evaluated on a holdout set and with 5-fold cross-validation, then retrained on the complete processed dataset and exported as `laptop_price_model.pkl`.

The Django application uses this model to produce an estimated laptop value from validated user input. The result is based on the training dataset and is not a live retail price.